# 7.2 Bayesian Classifier based on Normal Distribution

In this second notebook of the object recognition chapter we will explore a popular family of classifiers that follow the generative approach, the Bayesian ones. These methods are based on the idea that, given a feature vector $\mathbf{x}$ characterizing an object, the Bayes' rule can be used to compute the posterior probability of each class $C_k$ and predict its class. Recall the Bayes' rule: $\\[10pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/bayes_rule.png" width="700"/>
    <figcaption>Fig. 1. Bayes' rule and the meaning of each of its terms.</figcaption>
</center>$\\[5pt]$

In this way, Bayesian classifiers build probabilistic models of how each class generates the features (the likelihoods $p(\mathbf{x}|C_k)$ and the priors $P(C_k)$) from certain training data, and use them to classify new objects.

In this notebook, we will explain:
- how Bayesian classifiers work (<a href="#721">section 7.2.1</a>) and,
- specifically, how to classify feature vectors assuming that the ones of each class follow a Gaussian (normal) distribution (<a href="#722">section 7.2.2</a>).

## Problem context - Traffic sign recognition

In the previous notebook, *AliquindoiCars* contacted us looking for a TSR technique to be integrated into a self-driving car. They provided us with some segmented images of traffic signs that their autonomous cars captured during test drivings. These images are located in `images/circles/` containing circled signs, `images/triangles` containing signs having triangular shapes and `images/squares` containing signs having square shapes.$\\[3pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/signs.jpg" width="400"/>
    <figcaption>Fig. 2. Examples of traffic signs that a TSR system must recognize.</figcaption>
</center>$\\[5pt]$

Previously, we extracted a feature vector from each image using Hu moments. Now, we will train a classifier using the feature vectors we saved in the previous notebook. For classification, there are many methods we can apply, such as [kNN algorithm](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm), [random forest](https://en.wikipedia.org/wiki/Random_forest), etc. In this notebook, we will explore and use a classical one, the Bayesian classifier!

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 07. Object Recognition'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
import scipy.interpolate
matplotlib.rcParams['figure.figsize'] = (8.0, 8.0)

images_path = './images/'
import sys
sys.path.append("..")
from utils.PlotEllipse import PlotEllipse

### 7.2.1 Bayesian classifier <a id="721"></a>

Given a feature vector $\mathbf{x} = [x_1, x_2, \ldots, x_N]^T$ describing an object, and a set of $M$ possible classes $C_1, \ldots, C_M$, the Bayesian classifier computes the **posterior probability** $P(C_k|\mathbf{x})$ of each class with the Bayes' rule (see Fig. 1):

$$P(C_k|\mathbf{x}) = \frac{p(\mathbf{x}|C_k)P(C_k)}{p(\mathbf{x})}, \qquad p(\mathbf{x}) = \sum_{j=1}^{M} p(\mathbf{x}|C_j)P(C_j)$$

Then, it assigns $\mathbf{x}$ to the class with the highest posterior probability:

$$\hat{C}(\mathbf{x}) = \operatorname*{arg\,max}_k P(C_k|\mathbf{x})$$

This is called the **MAP** (Maximum A Posteriori) rule. With equal costs for all classification errors, it minimizes the probability of error, since for a decision $C_k$:

$$P(\text{error}|\mathbf{x}, \text{decide } C_k) = 1 - P(C_k|\mathbf{x})$$

The simplest case of these classifiers is the **Naïve Bayes** one, which considers the strong (naïve) assumption that the features are conditionally independent of each other given the object class. That is, for example, if we are using circularity and elongation to describe objects, this classifier assumes that the value for both features is not related if the object class is known, e.g. a circle. For the visual system in our previous notebook, which is in charge of recognizing objects in a kitchen, this means that if it knows that an object is a spoon, then its circularity would be unrelated with its elongation (we could not say anything about its possible elongation given a certain circularity, for example).

Writing the Bayes' rule with the $N$ features:

$$P(C_k|x_1,\dots,x_N)= \frac{p(x_1,\dots,x_N|C_k)P(C_k)}{p(x_1,\dots,x_N)}$$

if the following naïve conditional independence assumption:

$$p(x_f|C_k,x_1,\dots,x_{f-1},x_{f+1},\dots,x_N)=p(x_f|C_k)$$

is considered for each feature $x_f$, then the rule can be simplified to:

$$P(C_k|x_1,\dots,x_N)=\frac{P(C_k)\prod_{f=1}^N p(x_f|C_k)}{p(x_1,\dots,x_N)}$$

Given that $p(x_1,\dots,x_N)$ does not depend on $C_k$, and that we are looking for the class that has the highest posterior probability, the following classification rule is considered:

$$P(C_k|x_1,\dots,x_N) \propto P(C_k) \prod_{f=1}^{N} p(x_f|C_k)$$

$$\hat{C}(\mathbf{x}) = \operatorname*{arg\,max}_k P(C_k) \prod_{f=1}^{N} p(x_f|C_k)$$

Summarizing, the Bayesian classifier assigns $\mathbf{x}$ to the class $C_k$ that has the highest posterior probability $P(C_k|\mathbf{x})$ (the more probable, the less probability of making a mistake), that is, it makes a **MAP** prediction.

Consider the following posterior probabilities $P(C_k|x)$ for two classes (in this case $\mathbf{x}$ contains just one feature, $x$):$\\[3pt]$

In [ ]:
x = np.array([0, 1, 2, 3])
pC1_given_x = np.array([0.3, 0.4, 0.8, 0.9])
pC2_given_y = np.array([0.7, 0.6, 0.2, 0.1])

x_new = np.linspace(0, 3, 300)
a_BSpline = scipy.interpolate.make_interp_spline(x, pC1_given_x, bc_type="natural")
b_BSpline = scipy.interpolate.make_interp_spline(x, pC2_given_y, bc_type="natural")
pC1_given_x_new = a_BSpline(x_new)
pC2_given_y_new = b_BSpline(x_new)

plt.plot(x_new, pC1_given_x_new, label='$P(C_1|x)$')
plt.plot(x_new, pC2_given_y_new, label='$P(C_2|x)$')
p1, p2 = a_BSpline(0.5), b_BSpline(0.5)
plt.plot([0.5,0.5],[0, p1],'r'); plt.plot(0.5,p1,'or', label="$P(C_1|x=0.5)$")
plt.plot([0.5,0.5],[p1, p2],'g'); plt.plot(0.5,p2,'og', label="$P(C_2|x=0.5)$")

plt.ylabel('$P(C_k|x)$')
plt.xlabel('$x$')
plt.ylim([0,1])
plt.xlim([0,3])
plt.legend();

This MAP classification corresponds to assigning the object to the class $C_k$ with the highest value. An object characterized by $x=0.5$ is assigned to $C_2$, the class with the highest $P(C_k|x=0.5)$ value.

Depending on the application, it is convenient to consider a **reject region**, where no class is probable enough and no decision is made (e.g. $P(C_1|x) = P(C_2|x) = 0.5$). That is, given a probability threshold $\theta$:

$$\max_k P(C_k|x) < \theta \ \Rightarrow \ \text{reject}$$

Notice that $\max_k P(C_k|x) \geq 1/M$, so $\theta$ must be greater than $1/M$ (e.g. $\theta > 0.5$ for two classes) for the reject region to exist. The following code illustrates this for two classes, where neither posterior exceeds the threshold $\theta$ inside the reject region:$\\[3pt]$

In [ ]:
plt.plot(x_new, pC1_given_x_new, label='$P(C_1|x)$')
plt.plot(x_new, pC2_given_y_new, label='$P(C_2|x)$')

plt.ylabel('$P(C_k|x)$')
plt.xlabel('$x$')
plt.ylim([0,1])
plt.xlim([0,3])
plt.legend();

theta = 0.6 # threshold

# Reject where no posterior probability exceeds the threshold
reject = np.maximum(pC1_given_x_new, pC2_given_y_new) < theta
plt.plot([0, 3],[theta, theta],'g--')
plt.fill_between(x_new, 0, 1, where=reject, color='g', alpha=0.15)
plt.text(1.1, 0.1, 'Reject \n region');

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now that you have notions about the reject region, **answer the following questions**:

- Which class would an object with $x=0.5$ belong to?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Which class would an object with $x=1.2$ belong to?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Would the threshold $\theta$ be the same in any application?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### Building discriminant functions

In the same way as we built linear and generalized discriminant functions in the previous notebook, those functions can be also defined to design a Bayesian classifier. For that, the goal is to obtain a discriminant function $d_k(\mathbf{x})$ for each class $C_k$, such that $d_i(\mathbf{x}) > d_j(\mathbf{x})$ whenever $P(C_i|\mathbf{x}) > P(C_j|\mathbf{x})$, so that the general decision rule $\hat{C}(\mathbf{x}) = \operatorname*{arg\,max}_k P(C_k|\mathbf{x})$ becomes $\hat{C}(\mathbf{x}) = \operatorname*{arg\,max}_k d_k(\mathbf{x})$. Let's design them step by step!

From the Bayes' rule we have that:$\\[5pt]$

$$d_k(\mathbf{x}) = P(C_k|\mathbf{x}) = \frac{p(\mathbf{x}|C_k)P(C_k)}{p(\mathbf{x})}$$

This is, Bayes' rule defines a way to compute the posterior probability $P(C_k|\mathbf{x})$ of a feature vector $\mathbf{x}$ for each class $C_k$. We can further simplify this function, since any function that keeps the same ordering across classes is an equivalent discriminant function:$\\[5pt]$

$$
\begin{aligned}
d_k(\mathbf{x}) &= P(C_k|\mathbf{x}) = \frac{p(\mathbf{x}|C_k)P(C_k)}{p(\mathbf{x})}\\
&\Big\Downarrow \ \ p(\mathbf{x}) \text{ does not depend on } C_k\\
d_k(\mathbf{x}) &= p(\mathbf{x}|C_k)P(C_k)\\
&\Big\Downarrow \ \ \ln \text{ is monotonic}\\
d_k(\mathbf{x}) &= \ln p(\mathbf{x}|C_k) + \ln P(C_k)\\
&\Big\Downarrow \ \ \text{if } P(C_i) = P(C_j)\; \forall i,j\\
d_k(\mathbf{x}) &= \ln p(\mathbf{x}|C_k)
\end{aligned}
$$

The third one is the **MAP discriminant**, and the log form is easier to compute and numerically more stable (sums instead of products). The last one, for equal priors, is called the **maximum-likelihood classifier**.

### 7.2.2 Gaussian classifier <a id="722"></a>

The different Bayesian classifiers differ in the probability distribution considered for modeling the class-conditional densities $p(\mathbf{x}|C_k)$. In this section we will cover the **Gaussian class-conditional model**, which assumes that the feature vectors of each class follow a Gaussian (normal) distribution:$\\[5pt]$

$$p(\mathbf{x}|C_k) = \mathcal{N}(\mathbf{x}; \boldsymbol{\mu}_k, \Sigma_k) = \frac{1}{(2\pi)^{N/2}|\Sigma_k|^{1/2}}\, e^{-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu}_k)^T\Sigma_k^{-1}(\mathbf{x}-\boldsymbol{\mu}_k)}$$

so two sets of parameters are considered for each class $C_k$ (we drop the index $k$ below for clarity):$\\[5pt]$

- A mean vector: $\boldsymbol{\mu}=[\mu_1 \ \mu_2 \ \dots \ \mu_f \ \dots \ \mu_N]^T$
- A covariance matrix:

  $$\Sigma = E[(\mathbf{x}-\boldsymbol{\mu}) \cdot (\mathbf{x}-\boldsymbol{\mu})^T] =
  \begin{bmatrix}
  \sigma_{1,1} & \cdots & \sigma_{1,f} & \cdots & \sigma_{1,N} \\
  \vdots & & \vdots & &  \vdots \\
  \sigma_{N,1} & \cdots & \sigma_{N,f} & \cdots & \sigma_{N,N}
  \end{bmatrix}$$

If the features are conditionally independent given $C_k$, $\Sigma$ is diagonal, which is the assumption done by the **Gaussian Naïve Bayes** classifier:

$$\Sigma =
\begin{bmatrix}
\sigma_{1,1} & \cdots & 0 & \cdots & 0 \\
\vdots & & \vdots & &  \vdots \\
0 & \cdots & 0 & \cdots & \sigma_{N,N}
\end{bmatrix}$$

In this notebook we will use full covariance matrices, so our classifier is not a naïve one.

The following code is just a review about how a Gaussian distribution with two variables depends on its two parameters $\boldsymbol{\mu}$ and $\Sigma$. Feel free to change such parameters and experience their influence!

In [ ]:
def plot_2d_gaussian(fig, rv, x, y, pos, position):
    """ Plot 2d contours of a 3D gaussian   
    """
    label = "rv" + str(position)
    position = str(23)+str(position)
    ax = fig.add_subplot(int(position))
    ax.contourf(x, y, rv.pdf(pos))    
    ax.text(-4,-4,label,bbox=dict(facecolor='white', alpha=0.5))
    ax.set_aspect('equal')

from scipy.stats import multivariate_normal
x, y = np.mgrid[-5:5:.01, -5:5:.01]
pos = np.dstack((x, y))

# Define 6 different Gaussian distributions 
mean1 = np.array([2.0, 1.0])
covar1 = np.array([[0.5, 0.0],[0.0, 0.5]])
rv1 = multivariate_normal(mean1, covar1)

mean2 = np.array([-2.0, 2.0])
covar2 = np.array([[0.3, 0.0],[0.0, 1.8]])
rv2 = multivariate_normal(mean2, covar2)

mean3 = np.array([1.0, 0.0])
covar3 = np.array([[0.8, 0.7],[0.7, 1.3]])
rv3 = multivariate_normal(mean3, covar3)

mean4 = np.array([2.0, -1.6])
covar4 = np.array([[2.0, 0.0],[0.0, 2.0]])
rv4 = multivariate_normal(mean4, covar4)

mean5 = np.array([0.0, 0.0])
covar5 = np.array([[4.0, 0.0],[0.0, 1.8]])
rv5 = multivariate_normal(mean5, covar5)

mean6 = np.array([0.0, 2.0])
covar6 = np.array([[3.9, -0.5],[-0.5, 1.1]])
rv6 = multivariate_normal(mean6, covar6)

# Show the contours of the 3D Gaussians
fig = plt.figure()
plot_2d_gaussian(fig,rv1,x,y,pos,1)
plot_2d_gaussian(fig,rv2,x,y,pos,2)
plot_2d_gaussian(fig,rv3,x,y,pos,3)
plot_2d_gaussian(fig,rv4,x,y,pos,4)
plot_2d_gaussian(fig,rv5,x,y,pos,5)
plot_2d_gaussian(fig,rv6,x,y,pos,6)

### Designing the discriminant function

It is time to see how we could define the discriminant function $d_k(\mathbf{x})$ for this classifier:

$$
\begin{aligned}
d_k(\mathbf{x}) &=\ln P(C_k) + \ln p(\mathbf{x}|C_k) = \ln P(C_k) + \ln \frac{1}{(2\pi)^{N/2}|\Sigma_k|^{1/2}}e^{-\frac{1}{2}\overbrace{(\mathbf{x}-\boldsymbol{\mu}_k)^T\Sigma_k^{-1}(\mathbf{x}-\boldsymbol{\mu}_k)}^{\text{Squared Mahalanobis distance } D_k^2(\mathbf{x})}}\\[7pt]
    &=\ln P(C_k) - \frac{N}{2}\ln 2\pi - \frac{1}{2}\ln |\Sigma_k| - \frac{1}{2} D_k^2(\mathbf{x})\\[7pt]
    &=\ln P(C_k) - \frac{1}{2}\left[\underbrace{N \ln 2\pi}_{\text{constant}} +  \ln |\Sigma_k| + D_k^2(\mathbf{x})\right] \\[7pt]
&\Big\Downarrow \ \ \text{the constant term can be removed}\\[7pt]
d_k(\mathbf{x}) &=\ln P(C_k) - \frac{1}{2}\left[\ln |\Sigma_k| + D_k^2(\mathbf{x})\right]
\end{aligned}
$$

Expanding the squared Mahalanobis distance, we can see that the resulting **discriminant function is quadratic**:

$$
d_k(\mathbf{x})
=
-\frac{1}{2}\underbrace{\mathbf{x}^T \Sigma_k^{-1}\mathbf{x}}_{\text{Quadratic term}}
+
\underbrace{\mathbf{x}^T \Sigma_k^{-1}\boldsymbol{\mu}_k}_{\text{Linear term}}
+
\underbrace{\ln P(C_k) - \frac{1}{2} \left[ \ln |\Sigma_k| + \boldsymbol{\mu}_k^T \Sigma_k^{-1} \boldsymbol{\mu}_k \right]}_{\text{Independent term } b_k}
$$

That is, $d_k(\mathbf{x}) = \mathbf{x}^T Q_k \mathbf{x} + \mathbf{x}^T \mathbf{w}_k + b_k$, with $Q_k = -\frac{1}{2}\Sigma_k^{-1}$ and $\mathbf{w}_k = \Sigma_k^{-1}\boldsymbol{\mu}_k$. Since each class has its own covariance matrix $\Sigma_k$, this classifier is also known as **Quadratic Discriminant Analysis (QDA)**. Visually, the **decision boundaries are quadratic** (in 2D, conic sections like ellipses, parabolas or hyperbolas):

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/visual_normal_bayes.png" width="600"/>
    <figcaption>Fig. 3. Decision boundaries of a Bayesian classifier for three classes with normal distributions.</figcaption>
</center>$\\[5pt]$

### **<span style="color:green"><b><i>ASSIGNMENT 1a: Training the classifier</i></b></span>**

Now, we are going to implement this Gaussian classifier **using the Hu moments** computed in the previous notebook.

Training this classifier consists of estimating, for each class $C_k$, its **prior** $P(C_k)$, its **mean vector** $\boldsymbol{\mu}_k$ (dimension $(2,)$) and its **covariance matrix** $\Sigma_k$ (dimension $(2,2)$) from a training set of $n$ labeled pairs $(\mathbf{x}_i, c_i)$, $i=1,\ldots,n$. Using the one-hot labels $y_{i,k}$ ($y_{i,k}=1$ if $c_i = k$ and $0$ otherwise), the maximum likelihood estimates of these parameters are:

$$n_k = \sum_{i=1}^{n} y_{i,k}, \qquad P(C_k) = \frac{n_k}{n}, \qquad \boldsymbol{\mu}_k = \frac{1}{n_k}\sum_{i=1}^{n} y_{i,k}\,\mathbf{x}_i, \qquad \Sigma_k = \frac{1}{n_k}\sum_{i=1}^{n} y_{i,k}\,(\mathbf{x}_i-\boldsymbol{\mu}_k)(\mathbf{x}_i-\boldsymbol{\mu}_k)^T$$

being $n_k$ the number of samples in class $C_k$. Since we have $n_k=20$ samples of each shape, the priors are equal: $P(C_k)=1/3$.

In the previous notebook we proved that our TSR problem can be solved using only the first and second Hu moments. In this one **your first task** is:

- to load the firsts two Hu moments for the images from each class which, as commented, was computed in previous notebook (you can use [`np.load()`](https://numpy.org/doc/stable/reference/generated/numpy.load.html?highlight=load#numpy.load)). 
- Then, **compute the mean** (or centroid) and **covariance matrix** for each class. You can compute the covariance matrix of a set of points using [`np.cov()`](https://numpy.org/doc/stable/reference/generated/numpy.cov.html) with `bias=True`, so that it divides by $n_k$ as in the formula above (by default it divides by $n_k-1$).

In [ ]:
# Assignment 1a
#
# Load first 2 Hu moments of each class 
train_triangles = np.load("./data/hu_triangles.npy")[:,:2].T
train_circles = np.load("./data/hu_circles.npy")[:,:2].T
train_squares = np.load("./data/hu_squares.npy")[:,:2].T

# Compute covariance matrices
cov_triangles = np.cov(None, bias=None)
cov_circles = np.cov(None, bias=None)
cov_squares = np.cov(None, bias=None)

# Compute means
mean_triangles = None
mean_circles = None
mean_squares = None

print ('cov_triangles = \n' + str(cov_triangles))
print ('cov_circles = \n' + str(cov_circles))
print ('cov_squares = \n' + str(cov_squares))
print ('mean_triangles = ' + str(mean_triangles))
print ('mean_circles = ' + str(mean_circles))
print ('mean_squares = ' + str(mean_squares))

<span style="color:blue;">Expected output:</span>
```
cov_triangles = 
[[1.11723668e-06 2.13649677e-07]
 [2.13649677e-07 8.36673508e-08]]
cov_circles = 
[[9.70804293e-08 3.16570276e-08]
 [3.16570276e-08 1.03796090e-08]]
cov_squares = 
[[1.53433251e-06 4.95698797e-07]
 [4.95698797e-07 1.62296102e-07]]
mean_triangles = [0.19254439 0.00034407]
mean_circles = [1.59537533e-01 9.98702969e-05]
mean_squares = [0.16720802 0.00026462]
```

### **<span style="color:green"><b><i>ASSIGNMENT 1b: Defining the discriminant function</i></b></span>**

Your **next task** is to develop a method, named `discriminant_function()`, that computes the discriminant function $d_k(\mathbf{x})$ of a class $C_k$. The inputs have to be:

- `features`: feature vector $\mathbf{x}$ of dimension $N$ (number of features, 2 in our problem).
- `mu`: mean vector $\boldsymbol{\mu}_k$ of the class $C_k$.
- `cov`: covariance matrix $\Sigma_k$ with shape $(N,N)$ of the class $C_k$.
- `prior`: prior probability $P(C_k)$ of the class $C_k$.

The method should evaluate (then return) the discriminant function.

In [ ]:
# Assignment 1b
#
def discriminant_function(features, mu, cov, prior):
    """ Evaluates the discriminant function d_k(x) of a class C_k
    
        Args:
            features: feature vector of dimension N
            mu: mean vector of the class of which is being computed the probability
            cov: covariance matrix with shape (N,N) of the class
            prior: prior probability of class k

        Returns:
            dx: result of discriminant function
    """
    covinv = np.linalg.inv(cov) # Auxiliary variable
    dx = None # You can divide this computation in as many lines as you need
    
    return dx


**You can try** your function with the following test:

In [ ]:
f = np.array([0.5, 0.6])
mu = np.array([0.7, 0.9])
cov = np.array([[0.7, 0.3],[0.3, 0.9]])
prior = 0.5
d = discriminant_function(f, mu, cov, prior)

print ('d = ' + str(d))

<span style="color:blue;">Expected output:</span>
```
d = -0.4433874441813701
```

### **<span style="color:green"><b><i>ASSIGNMENT 1c: Testing the classifier</i></b></span>**

For testing our brand new classifier, we are going to classify some new images and check the results. *Note that the discriminant function is based on logarithms of probability densities, not a probability itself (values can be positive and negative, but the class that maximizes it is the same)*.

**What to do?** 
- Complete the auxiliary function `classify_image()` that:
    1. computes the Hu moments of a testing image `sign_image`, and   
    2. uses the discriminant function of each class to retrieve the highest output value. The class returning such a value would be the assigned one! $\\[10pt]$
- After completing such a function, in the code cell below it, call it for the `test_circle.png`, `test_square.png` and `test_triangle.png` images in order to classify them. 

*We assume that there is no prior information about any class, so $P(C_i) = P(C_j)\; \forall i,j$ (as in our training data). This can be interpreted as: while driving, we see the same number of circle, square and triangle shaped road signs.*


In [ ]:
# Assignment 1c
#
def image_moments(region):
    """ Compute moments of the external contour in a binary image.   
    
        Args:
            region: Binary image
                    
        Returns: 
            moments: dictionary containing all moments of the region
    """   
    
    # Get external contour
    contours,_ = cv2.findContours(region,cv2.RETR_EXTERNAL ,cv2.CHAIN_APPROX_NONE)
    cnt = contours[0]
    
    # Compute moments
    moments = cv2.moments(cnt)
    
    return moments

In [ ]:
def classify_image(sign_image):
    """ Classify a traffic sign image by its shape using a bayesian classifier   
    
        Args:
            sign_image: Binarized image
    """   
    
    # Compute Hu moments
    moments = image_moments(sign_image)
    hu = cv2.HuMoments(None).flatten()[None:None]
    
    # Classify circle test image
    prior = 1/3
    triangle = discriminant_function(None,None,None,None)
    circle = discriminant_function(None,None,None,None)
    square = discriminant_function(None,None,None,None)
    
    # Search the maximum
    classification = max([None,None,None])
    
    if classification == triangle:
        print("The sign is a triangle\n")
    elif classification == circle:
        print("The sign is a circle\n")
    else:
        print("The sign is a square\n")        
    
    return hu

In [ ]:
# Read images
test_circle = cv2.imread(images_path + "test_circle.png", 0)
test_triangle = cv2.imread(images_path + "test_triangle.png", 0)
test_square = cv2.imread(images_path + "test_square.png", 0)

# Classify them
print("Circle: ")
moments_circle = classify_image(None)
print("Triangle: ")
moments_triangle = classify_image(None)
print("Square: ")
moments_square = classify_image(None)

# Create figure
fig, ax = plt.subplots()
plt.axis([0.158, 0.197, -0.02, 0.02])

# Plot hu moments
plt.plot(train_triangles[0,:],train_triangles[1,:],'go')
plt.plot(train_circles[0,:],train_circles[1,:],'ro')
plt.plot(train_squares[0,:],train_squares[1,:],'bo')
plt.xlabel('x_1')
plt.ylabel('x_2')

# Plot testing data
plt.plot(moments_circle[0],moments_circle[1],'*m',markersize=12)
plt.plot(moments_triangle[0],moments_triangle[1],'*m',markersize=12)
plt.plot(moments_square[0],moments_square[1],'*m',markersize=12);

### **<span style="color:green"><b><i>ASSIGNMENT 1d: Analyzing covariances</i></b></span>**

Finally, we can see how this classifier divides the feature space showing the computed covariance ellipses. **You have to** complete the following code cell to make it works, showing the covariance ellipses of each class.

In [ ]:
# Create figure
fig, ax = plt.subplots()
plt.axis([0.14, 0.2, -0.03, 0.03])

# Plot hu moments
plt.plot(train_triangles[0,:],train_triangles[1,:],'go')
plt.plot(train_circles[0,:],train_circles[1,:],'ro')
plt.plot(train_squares[0,:],train_squares[1,:],'bo')

# Plot ellipses representing covariance matrices
PlotEllipse(fig, ax, np.vstack(None), None, 15, color='black')
PlotEllipse(fig, ax, np.vstack(None), None, 20, color='black')
PlotEllipse(fig, ax, np.vstack(None), None, 20, color='black')

fig.canvas.draw()

### Simplified Gaussian classifier: isotropic shared covariance

The classifier at hand can be simplified if the Euclidean distance is considered instead of the Mahalanobis one. This can be achieved if all the classes share the same **isotropic** covariance matrix (and the priors are equal):

$$\Sigma_k = \Sigma = \sigma^2 \cdot \mathbf{I} = \sigma^2 \begin{bmatrix}1 & \cdots & 0 \\ \vdots & \ddots & \vdots \\ 0 & \cdots & 1\end{bmatrix} \quad \forall k$$

In this way, the Gaussians of all classes are spherical and have the same size, and the decision boundaries are lines (hyperplanes in general):

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/natural_classifier.png" width="500"/>
    <figcaption>Fig. 4. Example of feature space with 3 classes characterized by Gaussian distributions with isotropic covariances. Black lines are decision boundaries.</figcaption>
</center>$\\[5pt]$

In this case, the discriminant function can be simplified:$\\[6pt]$

$$d_k(\mathbf{x}) = -\frac{1}{2\sigma^2}(\mathbf{x}-\boldsymbol{\mu}_k)^T(\mathbf{x}-\boldsymbol{\mu}_k) = -\frac{1}{2\sigma^2}\|\mathbf{x}-\boldsymbol{\mu}_k\|^2$$

Since $\sigma$ is the same for all classes, we can use $d_k(\mathbf{x}) = -\|\mathbf{x}-\boldsymbol{\mu}_k\|^2$. Moreover, expanding it, the quadratic term $\mathbf{x}^T\mathbf{x}$ is the same for all classes, so it can be removed and the decision boundaries become linear. Thereby, classification becomes the **minimum Euclidean distance to the centroid** (also known as nearest-centroid classifier):

$$\hat{C}(\mathbf{x}) = \arg\min_k \|\mathbf{x}-\boldsymbol{\mu}_k\|^2$$

### **<span style="color:green"><b><i>ASSIGNMENT 2: Playing with isotropic covariance matrices</i></b></span>**


**What to do?** Repeat the previous steps but using isotropic covariance matrices. Recall that [`np.eye()`](https://numpy.org/doc/stable/reference/generated/numpy.eye.html) defines an identity matrix. 

In [ ]:
# Assignment 2
#
def discriminant_function_isotropic(features, mu):
    """ Evaluates the discriminant function of a Gaussian classifier using isotropic covariances
    
        Args:
            features: feature vector of dimension N
            mu: mean vector of the class of which is being computed the probability

        Returns:
            dx: result of discriminant function
    """
    dx = None
    return dx

In [ ]:
def classify_image_isotropic(sign_image):
    """ Classify a traffic sign image by its shape using a bayesian classifier   
    
        Args:
            sign_image: Binarized image
    """   
    
    # Compute Hu moments
    moments = image_moments(sign_image)
    hu = cv2.HuMoments(None).flatten()[None:None]
    
    # Classify circle test image
    triangle = discriminant_function_isotropic(None,None)
    circle = discriminant_function_isotropic(None,None)
    square = discriminant_function_isotropic(None,None)
    
    # Search the maximum
    classification = max([None,None,None])
    
    if classification == triangle:
        print("The sign is a triangle\n")
    elif classification == circle:
        print("The sign is a circle\n")
    else:
        print("The sign is a square\n")

In [ ]:
# Read images
test_circle = cv2.imread(images_path + "test_circle.png", 0)
test_triangle = cv2.imread(images_path + "test_triangle.png", 0)
test_square = cv2.imread(images_path + "test_square.png", 0)

# Classify them
print("Circle: ")
classify_image_isotropic(None)
print("Triangle: ")
classify_image_isotropic(None)
print("Square: ")
classify_image_isotropic(None)

**Complete the following code** to observe the isotropic covariance matrices. 

In [ ]:
# Create figure
fig, ax = plt.subplots()
plt.axis([0.14, 0.2, -0.03, 0.03])

# Plot hu moments
plt.plot(train_triangles[0,:],train_triangles[1,:],'go')
plt.plot(train_circles[0,:],train_circles[1,:],'ro')
plt.plot(train_squares[0,:],train_squares[1,:],'bo')

# Plot ellipses representing covariance matrices
PlotEllipse(fig, ax, np.vstack(None), None, 0.003, color='black')
PlotEllipse(fig, ax, np.vstack(None), None, 0.003, color='black')
PlotEllipse(fig, ax, np.vstack(None), None, 0.003, color='black')

fig.canvas.draw()

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now that you are an expert concerning the Bayesian classifier, **answer the following questions**:

- Considering the classifier that you implemented in assignment 1c, to which class would be assigned an object with $\mathbf{x}=[0.17, -0.01]^T$? And considering the one in assignment 2?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- What are the pros and cons of using isotropic covariances?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- In what type of problems could isotropic matrices be used?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- In your opinion, is it worth to consider a Bayes classifier when dealing with this problem? In which situations could this classifier show its potential?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## Conclusion

Awesome! You now know how to design a classifier for previously segmented and characterized objects. Note that for more complex shapes, you can use the **7 Hu moments instead of the two that we used**. We reduced their number just for visualization and simplicity purposes.

In this notebook you have learned to:

- construct a Bayesian classifier where the features of each class follow a Gaussian distribution (QDA), and apply it to a real problem,
- build a simplified classifier where an isotropic shared covariance is assumed (minimum Euclidean distance to the centroid), and
- improve a classifier (if needed) using reject regions.

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>